In [0]:
# =============================================================================
# Notebook: 01_Historical_Load_BQ_to_S3 (Lakehouse Federation - Optimized)
# Strategy : Native Spark parallelism via range-partitioned reads.
#            No ThreadPoolExecutor — Spark already distributes work across
#            executors. One BigQuery scan per date-range partition, pushed
#            down as server-side filters by the federation connector.
# Compute  : Works on both Serverless and Job Cluster (auto-detects).
# =============================================================================

# --------------------------------------------------------------------------
# 0. Performance Configuration
#    All settings are safe for both Serverless and Job Cluster.
#    On Serverless, cluster-level JVM flags are ignored gracefully.
# --------------------------------------------------------------------------
# AQE: lets Spark coalesce shuffle partitions and pick better join strategies
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# S3 write optimizations
# EmrOptimizedSparkFileOutputCommitter is the fastest committer for S3
# (uses multipart upload, no rename cost). Falls back safely if not on EMR.
spark.conf.set("spark.hadoop.fs.s3a.fast.upload", "true")
spark.conf.set("spark.hadoop.fs.s3a.multipart.size", "134217728")          # 128 MB parts
spark.conf.set("spark.hadoop.fs.s3a.multipart.threshold", "134217728")
spark.conf.set("spark.hadoop.fs.s3a.threads.max", "64")
spark.conf.set("spark.hadoop.fs.s3a.connection.maximum", "64")
spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")  # skip rename step

# Parquet write tuning
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")
spark.conf.set("spark.sql.parquet.mergeSchema", "false")          # skip schema merges on read
spark.conf.set("spark.sql.parquet.filterPushdown", "true")

# Shuffle: start high, AQE will coalesce down automatically
spark.conf.set("spark.sql.shuffle.partitions", "400")

# --------------------------------------------------------------------------
# 1. Widgets
# --------------------------------------------------------------------------
dbutils.widgets.text("customer_id",      "cust_001")
dbutils.widgets.text("object_name",      "events")
dbutils.widgets.text("source_system",    "bigquery")
dbutils.widgets.text("bucket_path",      "s3://hgi-databricks-data-lakehouse-dev/")
dbutils.widgets.text("bq_catalog_table", "`bigquery-connection_catalog`.`v4c_bigquery_dataset`.`event_raw`")
dbutils.widgets.text("parallelism",      "32")     # number of Spark parallel range tasks

customer_id      = dbutils.widgets.get("customer_id")
object_name      = dbutils.widgets.get("object_name")
source_system    = dbutils.widgets.get("source_system")
bucket_path      = dbutils.widgets.get("bucket_path").rstrip("/")
bq_catalog_table = dbutils.widgets.get("bq_catalog_table")
parallelism      = int(dbutils.widgets.get("parallelism"))

from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

now  = datetime.utcnow()
yyyy, mm, dd, hh = now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H")

historical_s3_path = f"{bucket_path}/landing-zone/{source_system}/{customer_id}/{object_name}/historical/{yyyy}/{mm}/{dd}/{hh}"

print(f"=== Historical Load [Federation] ===")
print(f"Customer  : {customer_id}")
print(f"Object    : {object_name}")
print(f"Table     : {bq_catalog_table}")
print(f"S3 Path   : {historical_s3_path}")
print(f"Workers   : {parallelism}")

# --------------------------------------------------------------------------
# 2. Discover data bounds — single lightweight aggregation pushed to BQ
# --------------------------------------------------------------------------
bounds = (
    spark.table(bq_catalog_table)
         .select(
             F.min("event_timestamp").alias("min_ts"),
             F.max("event_timestamp").alias("max_ts")
         )
         .collect()[0]
)

start_ts, end_ts = bounds["min_ts"], bounds["max_ts"]
if start_ts is None:
    print("No data found in source. Exiting.")
    dbutils.notebook.exit("0")

print(f"Data range: {start_ts}  →  {end_ts}")

# --------------------------------------------------------------------------
# 3. Build range partitions as a Spark DataFrame
#    We create (range_start, range_end) pairs, then use them to drive
#    parallelism natively inside Spark — no Python threads needed.
#
#    For 1-3 years of data we use MONTHLY chunks (12-36 partitions).
#    Adjust `parallelism` widget to match your cluster size.
# --------------------------------------------------------------------------
total_seconds = int((end_ts - start_ts).total_seconds())
# Aim for `parallelism` roughly equal chunks
chunk_seconds = max(total_seconds // parallelism, 3600)   # minimum 1-hour chunks

intervals = []
cur = start_ts
while cur < end_ts:
    nxt = min(cur + timedelta(seconds=chunk_seconds), end_ts + timedelta(seconds=1))
    intervals.append((cur.strftime("%Y-%m-%d %H:%M:%S"),
                      nxt.strftime("%Y-%m-%d %H:%M:%S")))
    cur = nxt

print(f"Divided into {len(intervals)} chunks (~{chunk_seconds//3600:.1f}h each).")

# --------------------------------------------------------------------------
# 4. Parallel read + write using Spark's native parallelism
#
#    Key insight: instead of Python threads calling spark.table() in a loop
#    (which serialises Spark jobs), we create one DataFrame per interval
#    with a pushdown filter, union them all, and do ONE write.
#    The federation connector will push each filter predicate to BigQuery
#    Storage API, giving us parallel server-side reads.
# --------------------------------------------------------------------------
# Build union of filtered DataFrames — lazy, no data moved yet
dfs = [
    spark.table(bq_catalog_table)
         .filter(
             F.col("event_timestamp").between(
                 F.lit(start_str).cast(TimestampType()),
                 F.lit(end_str).cast(TimestampType())
             )
         )
    for start_str, end_str in intervals
]

# unionByName avoids schema mis-match issues across partitions
from functools import reduce
df_full = reduce(lambda a, b: a.unionByName(b), dfs)

# Repartition to match parallelism so each task writes one file.
# Using event_timestamp for locality — keeps related data together.
df_full = df_full.repartition(parallelism * 2, "event_timestamp")

# --------------------------------------------------------------------------
# 5. Write — single pass, Spark handles all parallelism
# --------------------------------------------------------------------------
print("Writing to S3 ...")
(
    df_full
    .write
    .mode("overwrite")                          # idempotent for historical loads
    .option("compression", "snappy")
    .format("parquet")
    .save(historical_s3_path)
)
print(f"✅ Write complete → {historical_s3_path}")

# --------------------------------------------------------------------------
# 6. Watermark — use MERGE (atomic, no temp view needed)
# --------------------------------------------------------------------------
spark.sql("""
    CREATE TABLE IF NOT EXISTS ingestion_metadata.watermarks (
        source_system              STRING,
        object_name                STRING,
        last_processed_timestamp   TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'false',
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")

# Subtract 1 minute for overlap safety (same as original logic)
watermark_ts = end_ts - timedelta(minutes=1)

spark.sql(f"""
    MERGE INTO ingestion_metadata.watermarks AS tgt
    USING (
        SELECT
            '{source_system}' AS source_system,
            '{object_name}'   AS object_name,
            TIMESTAMP('{watermark_ts}') AS last_processed_timestamp
    ) AS src
    ON tgt.source_system = src.source_system
   AND tgt.object_name   = src.object_name
    WHEN MATCHED THEN UPDATE SET
        tgt.last_processed_timestamp = src.last_processed_timestamp
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"✅ Watermark set to {watermark_ts}  (end_ts minus 1 min).")
dbutils.notebook.exit("Success")
